# Multi-scale figure: from the cosmic web to quenched dusty galaxies

3-level zoom connecting the full SIMBA 25 Mpc simulation volume to the CGM
and individual galactic structure of quenched, dust-rich galaxies.

**Primary selection (Caesar catalog) — dust-rich:**
- sSFR < 10⁻¹⁰ yr⁻¹ — effectively quenched
- M★ > 5 × 10¹⁰ M☉
- M_dust / M★ > 10⁻⁴ — dust-rich despite quenching

**Contrast selection — dust-poor:**
- Same quenched (sSFR < 10⁻¹⁰ yr⁻¹) and mass (M★ > 5 × 10¹⁰ M☉) cuts
- M_dust / M★ < 10⁻⁵ — dust-poor quenched galaxies for comparison
- The `N_DUST_POOR` most massive are appended as extra rows

**Figure layout (per selected galaxy):**
| Column 0 | Column 1 | Column 2 | Column 3 |
|---|---|---|---|
| Full 25 Mpc box *(spans rows)* | CGM — 100 kpc | Galaxy — 30 kpc | SED |

In [30]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle, Circle
from matplotlib.lines import Line2D

import h5py
import gc
import os

import yt
import caesar
import sphviewer as sph
from sphviewer.tools import QuickView, Blend

from simbanator.io.simba import Simulation
from simbanator.visualization.rendering import (
    get_normalized_image, apply_dust_screen, find_rot_ax,
)

plt.rcParams.update({
    'savefig.facecolor': 'k',
    'figure.facecolor': 'k',
    'axes.facecolor': 'k',
    'text.color': 'w',
    'axes.edgecolor': 'w',
    'axes.labelcolor': 'w',
    'xtick.color': 'w',
    'ytick.color': 'w',
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'cm',
    'font.size': 14,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
})

## 1 – Configuration

In [ ]:
# ── Snapshot ──────────────────────────────────────────────────────────────────
SNAP = 129   # z ≈ 0.4 for m25n512

# ── Selection criteria ────────────────────────────────────────────────────────
SFR_MAX       = 1e-10   # yr⁻¹   (quenched: sSFR = SFR / Mstar)
MSTAR_MIN     = 5e10    # Msun
DUST_FRAC_MIN = 1e-4    # Mdust / Mstar  (dust-rich primary sample)

# Dust-poor contrast sample: same quenched + mass cuts, but Mdust/Mstar below
# DUST_FRAC_POOR_MAX. We keep only the N_DUST_POOR most massive of these and
# append them to the primary selection (extra rows in the figure).
DUST_FRAC_POOR_MAX = 1e-5   # Mdust / Mstar
N_DUST_POOR        = 3      # how many dust-poor galaxies to add

# ── Scale half-extents (physical kpc) ─────────────────────────────────────────
EXT_GALAXY = 30    # ~60 kpc diameter  (resolved galaxy)
EXT_CGM    = 100   # ~200 kpc diameter (CGM environment)
# Full-box half-extent is derived from the snapshot header

# ── Render resolution ─────────────────────────────────────────────────────────
PIX_BOX    = 800
PIX_CGM    = 600
PIX_GALAXY = 500

# Subsample stride for the full-box panel (every N-th particle; memory trade-off)
BOX_STRIDE = 8

# ── SED paths ─────────────────────────────────────────────────────────────────
# Expected layout: <base>/<dust_on|dust_off>/powderday_sed_out/snap_NNN/gal_ID/...
SED_BASE    = '/mnt/home/glorenzon/analize_simba_cgm/output/cis25/sed'
SED_DIR_ON  = f'{SED_BASE}/dust_on/powderday_sed_out'
SED_DIR_OFF = f'{SED_BASE}/dust_off/powderday_sed_out'

def sed_path(run_dir, snap, gal_id):
    return (f'{run_dir}/snap_{snap:03d}/gal_{gal_id}'
            f'/snap{snap:03d}.galaxy{gal_id:06d}.rtout.sed')

# ── Load simulation handles ────────────────────────────────────────────────────
sb       = Simulation('cis25')
snapfile = sb.get_sim_file(SNAP)
catfile  = sb.get_caesar_file(SNAP)
z_snap   = sb.get_z_from_snap(SNAP)

print(f"Snapshot {SNAP}: z = {z_snap:.3f}")
print(f"Snapshot : {snapfile}")
print(f"Catalog  : {catfile}")

## 2 – Galaxy selection from Caesar

In [ ]:
obj = caesar.load(catfile)
a   = obj.simulation.scale_factor

# ── Primary sample: quenched, massive, dust-rich ─────────────────────────────
print(f"Searching {len(obj.galaxies)} galaxies for the dust-rich sample:")
print(f"  sSFR < {SFR_MAX:.0e}  yr⁻¹")
print(f"  M★   > {MSTAR_MIN:.1e}  Msun")
print(f"  Mdust/M★ > {DUST_FRAC_MIN:.0e}")
print()

selected_ids = []
for i, g in enumerate(obj.galaxies):
    mstar = float(g.masses['stellar'])
    mdust = float(g.masses['dust'])
    sfr   = float(g.sfr)
    if mstar > 0 and mdust > 0:
        if sfr / mstar < SFR_MAX and mstar > MSTAR_MIN and mdust / mstar > DUST_FRAC_MIN:
            selected_ids.append(i)
            print(f"  [galaxy {i:5d}]  "
                  f"M★ = {mstar:.2e}  "
                  f"Mdust = {mdust:.2e}  "
                  f"Mdust/M★ = {mdust/mstar:.2e}  "
                  f"SFR = {sfr:.2e}  Msun/yr")

print(f"\n→ {len(selected_ids)} dust-rich galaxy/galaxies: {selected_ids}")

# ── Contrast sample: same quenched + mass cuts, but dust-poor ────────────────
# Galaxies are returned in descending stellar-mass order, so taking the first
# N_DUST_POOR matches the mass range of the dust-rich sample as closely as
# possible. mdust may legitimately be ~0 here, so we don't require mdust > 0.
print(f"\nSearching for the dust-poor contrast sample:")
print(f"  sSFR < {SFR_MAX:.0e}  yr⁻¹")
print(f"  M★   > {MSTAR_MIN:.1e}  Msun")
print(f"  Mdust/M★ < {DUST_FRAC_POOR_MAX:.0e}")
print()

dust_poor_ids = []
for i, g in enumerate(obj.galaxies):
    mstar = float(g.masses['stellar'])
    mdust = float(g.masses['dust'])
    sfr   = float(g.sfr)
    if mstar > 0:
        if sfr / mstar < SFR_MAX and mstar > MSTAR_MIN and mdust / mstar < DUST_FRAC_POOR_MAX:
            dust_poor_ids.append(i)
            print(f"  [galaxy {i:5d}]  "
                  f"M★ = {mstar:.2e}  "
                  f"Mdust = {mdust:.2e}  "
                  f"Mdust/M★ = {mdust/mstar:.2e}  "
                  f"SFR = {sfr:.2e}  Msun/yr")

# Keep the most massive few and append them to the primary selection
dust_poor_ids = dust_poor_ids[:N_DUST_POOR]
selected_ids += dust_poor_ids

print(f"\n→ {len(dust_poor_ids)} dust-poor galaxy/galaxies added: {dust_poor_ids}")
print(f"→ {len(selected_ids)} total galaxies selected: {selected_ids}")

## 3 – Filter particles

In [34]:
from simbanator.analysis.particles import extract_particles

extract_particles(
    obj,
    snapfile,
    snap=SNAP,
    galaxy_ids=selected_ids,
    sim_name=sb.name,
    overwrite=True,
    prefix='m25n512',
)

Reading snapshot once for batch extraction: /mnt/home/share/simbas/SIMBA_25/snap_m25n512_129.hdf5
gal 1 PartType0 particles: 1670
gal 1 PartType4 particles: 178108


KeyboardInterrupt: 

## 4 – Write Powderday masters

In [ ]:
from simbanator.sed.makesed import MakeSED

snaps_25 = np.array([SNAP] * len(selected_ids), dtype=int)
ids_25   = np.array(selected_ids, dtype=int)

GVFS_BASE     = "/run/user/1000/gvfs/sftp:host=slurmusrint2.cis.gov.pl,user=glorenzon"
REMOTE_HOME25 = os.path.join(GVFS_BASE, "mnt/home/glorenzon/analize_simba_cgm")

hydro_dir_base_25 = os.path.join(os.getcwd(), 'output', sb.name, 'filtered_particles')
output_dir_25     = os.path.join(REMOTE_HOME25, 'output', sb.name, 'sed')
selection_file_25 = 'selection_simba25_multiscale'

print(f"Using output_dir: {output_dir_25}")

# Dust-on run
makesed_on = MakeSED(sb, nnodes=1, model_run_name='dust_on',
                     hydro_dir_base=hydro_dir_base_25,
                     selection_file=selection_file_25,
                     output_dir=output_dir_25,
                     run_tag='dust_on')
makesed_on.selection_gals(snaps=snaps_25, galaxyID=ids_25)
makesed_on.create_master('cluster', 'plist', radius=None,
                         partition='INTEL_PHI', prefix='m25n512',
                         paramf='parameters_master.py', snaps_to_run=None)
print("dust_on master scripts written.")

# Dust-off run
makesed_off = MakeSED(sb, nnodes=1, model_run_name='dust_off',
                      hydro_dir_base=hydro_dir_base_25,
                      selection_file=selection_file_25,
                      output_dir=output_dir_25,
                      run_tag='dust_off')
makesed_off.selection_gals(snaps=snaps_25, galaxyID=ids_25)
makesed_off.create_master('cluster', 'plist', radius=None,
                          partition='INTEL_PHI', prefix='m25n512',
                          paramf='parameters_master.py', snaps_to_run=None)
print("dust_off master scripts written.")

In [ ]:
# ── Run Powderday ONLY on the newly added dust-poor galaxies ──────────────────
# The SLURM master job is an array that runs one task per line of ids.txt, so the
# only lever for "just these galaxies" is which IDs go into the selection. We
# build a SEPARATE selection containing just `dust_poor_ids`, but with the SAME
# output_dir + run_tag, so the new SEDs land in the SAME powderday_sed_out tree
# (gal_<id>/...). This rewrites snap_<SNAP>/ids.txt and master.snap<SNAP>.job to
# cover only the new galaxies; the already-finished dust-rich SEDs on disk are
# left untouched.
#
# ⚠️  ORDER MATTERS: re-running the full create_master in section 4 puts all 15
#     IDs back into ids.txt. Submit the job written by THIS cell first.
#
# Prerequisite: the filtered-particle .h5 for each new galaxy must exist. If you
# haven't extracted them yet, run section 3 (Filter particles) with
# `galaxy_ids=dust_poor_ids` (not the full list) so you don't re-extract the 12.

new_ids   = np.array(dust_poor_ids, dtype=int)
new_snaps = np.full(len(new_ids), SNAP, dtype=int)
print(f"Building Powderday masters for {len(new_ids)} dust-poor galaxies only: {new_ids.tolist()}")

for tag in ('dust_on', 'dust_off'):
    ms = MakeSED(sb, nnodes=1, model_run_name=tag,
                 hydro_dir_base=hydro_dir_base_25,
                 selection_file='selection_simba25_multiscale_dustpoor',
                 output_dir=output_dir_25,
                 run_tag=tag)
    ms.selection_gals(snaps=new_snaps, galaxyID=new_ids)
    ms.create_master('cluster', 'plist', radius=None,
                     partition='INTEL_PHI', prefix='m25n512',
                     paramf='parameters_master.py', snaps_to_run=None)
    job = os.path.join(ms.model_dir_base, f'snap_{SNAP:03d}', f'master.snap{SNAP}.job')
    print(f"{tag}: master rebuilt for dust-poor galaxies only → sbatch {job}")

## 5 – Load particles

In [35]:
# ── Unit conversion from snapshot header ─────────────────────────────────────
with h5py.File(snapfile, 'r') as f:
    hdr      = dict(f['Header'].attrs)
    h5_h     = float(hdr['HubbleParam'])
    h5_a     = float(hdr['Time'])
    u_len    = float(hdr.get('UnitLength_in_cm',  3.085678e21))
    u_mass   = float(hdr.get('UnitMass_in_g',     1.989e43))
    box_code = float(hdr['BoxSize'])

kpc_in_cm   = 3.085678e21
msun_in_g   = 1.989e33
kpc_factor  = (u_len / kpc_in_cm) * h5_a / h5_h
msun_factor = (u_mass / msun_in_g) / h5_h

box_size_kpc   = box_code * kpc_factor
box_center_kpc = np.array([box_size_kpc / 2] * 3)

print(f"Box: {box_size_kpc:.0f} kpc  ({box_size_kpc/1000:.2f} Mpc physical at z={z_snap:.2f})")

# ── Per-galaxy: CGM-scale and galaxy-scale particles ─────────────────────────
gal_data = []

for gid in selected_ids:
    gal    = obj.galaxies[gid]
    center = gal.minpotpos.in_units('kpc').value
    L      = gal.rotation['gas_L']
    t, p   = find_rot_ax(L, spos='faceon')

    glist = np.sort(gal.glist)
    slist = np.sort(gal.slist)

    # Galaxy-member particles (used for 30 kpc view)
    with h5py.File(snapfile, 'r') as f:
        gal_gas_pos   = (f['PartType0/Coordinates'][glist] * kpc_factor).astype('f4')
        gal_gas_mass  = (f['PartType0/Masses'][glist]      * msun_factor).astype('f4')
        gal_dust_mass = (f['PartType0/Dust_Masses'][glist] * msun_factor).astype('f4')
        gal_star_pos  = (f['PartType4/Coordinates'][slist] * kpc_factor).astype('f4')
        gal_star_mass = (f['PartType4/Masses'][slist]      * msun_factor).astype('f4')

    # CGM-scale particles (all within EXT_CGM kpc of the galaxy center)
    ext_code    = EXT_CGM / kpc_factor
    center_code = center / kpc_factor
    with h5py.File(snapfile, 'r') as f:
        coords = f['PartType0/Coordinates'][:]
        mask   = np.all(np.abs(coords - center_code) < ext_code, axis=1)
        cgm_gas_pos  = (coords[mask] * kpc_factor).astype('f4')
        del coords
        masses = f['PartType0/Masses'][:]
        cgm_gas_mass = (masses[mask] * msun_factor).astype('f4')
        del masses, mask

        coords = f['PartType4/Coordinates'][:]
        mask   = np.all(np.abs(coords - center_code) < ext_code, axis=1)
        cgm_star_pos  = (coords[mask] * kpc_factor).astype('f4')
        del coords
        masses = f['PartType4/Masses'][:]
        cgm_star_mass = (masses[mask] * msun_factor).astype('f4')
        del masses, mask
    gc.collect()

    gal_data.append({
        'id': gid, 'center': center, 't': t, 'p': p,
        'gal_gas_pos':   gal_gas_pos,   'gal_gas_mass':  gal_gas_mass,
        'gal_star_pos':  gal_star_pos,  'gal_star_mass': gal_star_mass,
        'gal_dust_mass': gal_dust_mass,
        'cgm_gas_pos':   cgm_gas_pos,   'cgm_gas_mass':  cgm_gas_mass,
        'cgm_star_pos':  cgm_star_pos,  'cgm_star_mass': cgm_star_mass,
    })
    print(f"Galaxy {gid:5d}: {len(gal_gas_pos):,} gal-gas | {len(gal_star_pos):,} gal-stars")
    print(f"         CGM ({EXT_CGM} kpc): {len(cgm_gas_pos):,} gas | {len(cgm_star_pos):,} stars")

# ── Full-box particles: subsampled for memory efficiency ──────────────────────
print(f"\nLoading full-box data (stride = {BOX_STRIDE})…")
with h5py.File(snapfile, 'r') as f:
    box_gas_pos   = (f['PartType0/Coordinates'][::BOX_STRIDE] * kpc_factor).astype('f4')
    box_gas_mass  = (f['PartType0/Masses'][::BOX_STRIDE]      * msun_factor).astype('f4')
    box_star_pos  = (f['PartType4/Coordinates'][::BOX_STRIDE] * kpc_factor).astype('f4')
    box_star_mass = (f['PartType4/Masses'][::BOX_STRIDE]      * msun_factor).astype('f4')
gc.collect()
print(f"Full-box: {len(box_gas_pos):,} gas | {len(box_star_pos):,} stars  (1/{BOX_STRIDE} of total)")

Box: 26153 kpc  (26.15 Mpc physical at z=0.41)
Galaxy     1: 1,670 gal-gas | 178,108 gal-stars
         CGM (100 kpc): 45,379 gas | 222,202 stars
Galaxy     3: 4,671 gal-gas | 125,547 gal-stars
         CGM (100 kpc): 11,649 gas | 163,854 stars
Galaxy    10: 10,757 gal-gas | 63,313 gal-stars
         CGM (100 kpc): 45,707 gas | 101,274 stars
Galaxy    14: 9,177 gal-gas | 53,163 gal-stars
         CGM (100 kpc): 45,381 gas | 68,302 stars
Galaxy    15: 1,514 gal-gas | 51,573 gal-stars
         CGM (100 kpc): 5,945 gas | 58,020 stars
Galaxy    17: 1,403 gal-gas | 51,730 gal-stars
         CGM (100 kpc): 14,874 gas | 59,574 stars
Galaxy    24: 3,823 gal-gas | 39,778 gal-stars
         CGM (100 kpc): 18,673 gas | 63,540 stars
Galaxy    28: 2,930 gal-gas | 35,636 gal-stars
         CGM (100 kpc): 24,992 gas | 45,861 stars
Galaxy    31: 7,836 gal-gas | 33,894 gal-stars
         CGM (100 kpc): 14,733 gas | 39,295 stars
Galaxy    35: 2,149 gal-gas | 31,260 gal-stars
         CGM (100 kpc): 21,7

## 6 – Render at three scales

In [ ]:
def make_camera(center, extent, xsize, ysize, t, p):
    return sph.Camera(
        x=float(center[0]), y=float(center[1]), z=float(center[2]),
        r='infinity', t=float(t), p=float(p), roll=0,
        extent=[-extent, extent, -extent, extent],
        xsize=xsize, ysize=ysize,
    )

def render_image(pos, mass, camera):
    S = sph.Scene(sph.Particles(pos, mass), Camera=camera)
    R = sph.Render(S)
    return R.get_image(), R.get_extent()

def blend_gas_stars(gas_img, star_img,
                    dust_img=None, tau_max=1.5,
                    gas_cmap=cm.magma, star_cmap=cm.Greys_r,
                    gas_vmin=50, gas_vmax=99.5, gas_mode='log',
                    star_vmin=50, star_vmax=99.7, star_mode='log',
                    gamma=1.0):
    """Blend a gas + star SPH map into an RGBA composite.

    The normalisation is the whole story for dynamic range:
      * ``*_vmin`` is a PERCENTILE.  Raise it to clip the diffuse background
        to black — this is what stops the full-box / CGM panels from glowing.
      * ``*_vmax`` is the high-percentile anchor.  Pushing it toward 100 maps
        the very brightest pixel to white, so a sharply-peaked galaxy core
        saturates *less* (fewer pixels hit 1.0).
      * ``gamma`` > 1 darkens mid-tones after stretching; this is the main
        lever for "too white", because the ``Screen`` blend only brightens.
    """
    gas_n  = get_normalized_image(gas_img,  vmin=gas_vmin,  vmax=gas_vmax,  mode=gas_mode)
    star_n = get_normalized_image(star_img, vmin=star_vmin, vmax=star_vmax, mode=star_mode)
    if dust_img is not None:
        dust_n = get_normalized_image(dust_img, vmin=5, vmax=99, mode='log')
        gas_n  = apply_dust_screen(gas_n,  dust_n, tau_max)
        star_n = apply_dust_screen(star_n, dust_n, tau_max)
    if gamma != 1.0:
        gas_n  = np.clip(gas_n,  0, 1) ** gamma
        star_n = np.clip(star_n, 0, 1) ** gamma
    return Blend.Blend(star_cmap(star_n), gas_cmap(gas_n)).Screen()

# ── Per-scale normalisation knobs (tune these, not the function body) ─────────
# Large diffuse fields need a high vmin so the background goes black.
# gamma is the lever for de-whitening; raise it if a panel is still washed out.
NORM_BOX = dict(gas_vmin=65, gas_vmax=99.7, star_vmin=70, star_vmax=99.9, gamma=1.3)
NORM_CGM = dict(gas_vmin=58, gas_vmax=99.6, star_vmin=62, star_vmax=99.9, gamma=1.6)
NORM_GAL = dict(gas_vmin=20, gas_vmax=99.8, star_vmin=30, star_vmax=99.9, gamma=1.7)

# ── Full-box: top-down view (t=0, p=90 = looking along –z) ───────────────────
print("Rendering full 25 Mpc box…")
EXT_BOX = box_size_kpc / 2
cam_box = make_camera(box_center_kpc, EXT_BOX, PIX_BOX, PIX_BOX, t=0, p=90)
img_gas_box,  ext_box = render_image(box_gas_pos,  box_gas_mass,  cam_box)
img_star_box, _       = render_image(box_star_pos, box_star_mass, cam_box)
rgb_box = blend_gas_stars(img_gas_box, img_star_box, **NORM_BOX)
print("  done.")

# ── Per-galaxy: CGM (100 kpc) and galaxy (30 kpc) renders ────────────────────
# Both scales render the *dense region* particle set (cgm_*). The galaxy panel
# is just a 30 kpc zoom of the CGM. We deliberately do NOT use the Caesar
# galaxy-member set (gal_star_pos) here: those bound outskirt stars are sparse
# in 3D, so sphviewer's auto smoothing length blows up and smears them into
# multi-kpc bubbles — the puffy white ball artefact. The region set keeps
# neighbours close, so smoothing lengths stay small and the render is sharp.
for gd in gal_data:
    c, t, p = gd['center'], gd['t'], gd['p']

    print(f"Galaxy {gd['id']} — CGM ({EXT_CGM} kpc)…")
    cam_cgm = make_camera(c, EXT_CGM, PIX_CGM, PIX_CGM, t, p)
    img_gas_cgm,  ext_cgm = render_image(gd['cgm_gas_pos'],  gd['cgm_gas_mass'],  cam_cgm)
    img_star_cgm, _       = render_image(gd['cgm_star_pos'], gd['cgm_star_mass'], cam_cgm)
    gd['rgb_cgm'] = blend_gas_stars(img_gas_cgm, img_star_cgm, **NORM_CGM)
    gd['ext_cgm'] = ext_cgm

    print(f"Galaxy {gd['id']} — galaxy ({EXT_GALAXY} kpc)…")
    cam_gal = make_camera(c, EXT_GALAXY, PIX_GALAXY, PIX_GALAXY, t, p)
    img_gas_gal,  ext_gal = render_image(gd['cgm_gas_pos'],  gd['cgm_gas_mass'],  cam_gal)
    img_star_gal, _       = render_image(gd['cgm_star_pos'], gd['cgm_star_mass'], cam_gal)
    gd['rgb_gal'] = blend_gas_stars(img_gas_gal, img_star_gal, **NORM_GAL)
    gd['ext_gal'] = ext_gal
    print("  done.")

# ── Save individual CGM and galaxy panels right after rendering ───────────────
def _add_scalebar(ax, ext, bar_kpc, color='w', lw=2.5, fontsize=11, pad=0.06):
    dx = ext[1] - ext[0]; dy = ext[3] - ext[2]
    x0 = ext[0] + pad * dx;  y0 = ext[2] + pad * dy
    ax.plot([x0, x0 + bar_kpc], [y0, y0], lw=lw, color=color, solid_capstyle='butt')
    label = f'{bar_kpc:.0f} kpc' if bar_kpc >= 1 else f'{bar_kpc*1e3:.0f} pc'
    ax.text(x0 + bar_kpc / 2, y0 + 0.035 * dy,
            label, ha='center', va='bottom', color=color, fontsize=fontsize)

# One distinct, bright colour per selected galaxy (works for any N).
_GAL_COLORS = [mcolors.hsv_to_rgb((h, 0.85, 1.0))
               for h in np.linspace(0.0, 1.0, len(gal_data), endpoint=False)]
_out_dir = os.path.join(os.getcwd(), 'output', 'figures')
os.makedirs(_out_dir, exist_ok=True)

for _row, (_gd, _col) in enumerate(zip(gal_data, _GAL_COLORS)):
    _label = f'G{_row + 1}'
    _gobj  = obj.galaxies[_gd['id']]
    _mstar = float(_gobj.masses['stellar'])

    # ── CGM panel ─────────────────────────────────────────────────────────────
    _e = _gd['ext_cgm']
    _fig, _ax = plt.subplots(figsize=(6, 6), facecolor='k')
    _ax.imshow(_gd['rgb_cgm'], extent=_e, origin='lower', aspect='equal')
    _ax.set_xlim(_e[0], _e[1]); _ax.set_ylim(_e[2], _e[3])
    _ax.set_xticks([]); _ax.set_yticks([])
    for _sp in _ax.spines.values():
        _sp.set_edgecolor(_col); _sp.set_linewidth(1.5)
    _ax.add_patch(Rectangle(
        (-EXT_GALAXY, -EXT_GALAXY), 2 * EXT_GALAXY, 2 * EXT_GALAXY,
        lw=1.5, ec=_col, fc='none', ls='--'))
    _add_scalebar(_ax, _e, 50, color=_col, fontsize=12)
    _ax.set_title(
        rf'{_label}  CGM  ({EXT_CGM} kpc)  —  face-on',
        color=_col, fontsize=12, pad=5)
    _fig.tight_layout(pad=0.3)
    _p = os.path.join(_out_dir, f'cgm_{_label}_snap{SNAP}.pdf')
    _fig.savefig(_p, dpi=200, bbox_inches='tight', facecolor='k')
    print(f"Saved → {_p}")
    plt.close(_fig)

    # ── Galaxy panel ──────────────────────────────────────────────────────────
    _e = _gd['ext_gal']
    _fig, _ax = plt.subplots(figsize=(6, 6), facecolor='k')
    _ax.imshow(_gd['rgb_gal'], extent=_e, origin='lower', aspect='equal')
    _ax.set_xlim(_e[0], _e[1]); _ax.set_ylim(_e[2], _e[3])
    _ax.set_xticks([]); _ax.set_yticks([])
    for _sp in _ax.spines.values():
        _sp.set_edgecolor(_col); _sp.set_linewidth(1.5)
    _add_scalebar(_ax, _e, 10, color=_col, fontsize=12)
    _ax.set_title(
        rf'{_label}  ({EXT_GALAXY} kpc)  —  face-on   $M_\star = {_mstar:.1e}\,M_\odot$',
        color=_col, fontsize=12, pad=5)
    _fig.tight_layout(pad=0.3)
    _p = os.path.join(_out_dir, f'galaxy_{_label}_snap{SNAP}.pdf')
    _fig.savefig(_p, dpi=200, bbox_inches='tight', facecolor='k')
    print(f"Saved → {_p}")
    plt.close(_fig)

## 7 – Compose figure

In [ ]:
# ── Load SEDs (graceful fallback if files not found) ─────────────────────────
def load_sed(path, z):
    from hyperion.model import ModelOutput
    m = ModelOutput(path)
    wav, flux = m.get_sed(inclination='all', aperture=-1)
    return np.asarray(wav) * (1 + z), np.asarray(flux)

for gd in gal_data:
    gd['sed_wav'] = gd['sed_dust'] = gd['sed_nodust'] = None
    for key, run_dir in [('sed_dust', SED_DIR_ON), ('sed_nodust', SED_DIR_OFF)]:
        path = sed_path(run_dir, SNAP, gd['id'])
        try:
            wav, sed = load_sed(path, z_snap)
            gd['sed_wav'] = wav
            gd[key] = sed
            print(f"Galaxy {gd['id']}: {key} loaded.")
        except Exception as exc:
            print(f"Galaxy {gd['id']}: {key} not found — {exc}")

# ── Helpers ───────────────────────────────────────────────────────────────────
# One distinct, bright colour per selected galaxy (works for any N).
GAL_COLORS = [mcolors.hsv_to_rgb((h, 0.85, 1.0))
              for h in np.linspace(0.0, 1.0, len(gal_data), endpoint=False)]

def add_scalebar(ax, ext, bar_kpc, color='w', lw=2.5, fontsize=11, pad=0.06):
    dx = ext[1] - ext[0]; dy = ext[3] - ext[2]
    x0 = ext[0] + pad * dx;  y0 = ext[2] + pad * dy
    ax.plot([x0, x0 + bar_kpc], [y0, y0], lw=lw, color=color, solid_capstyle='butt')
    label = f'{bar_kpc:.0f} kpc' if bar_kpc >= 1 else f'{bar_kpc*1e3:.0f} pc'
    ax.text(x0 + bar_kpc / 2, y0 + 0.035 * dy,
            label, ha='center', va='bottom', color=color, fontsize=fontsize)

# ── Figure layout: N rows × 4 cols, col 0 spans all rows ─────────────────────
N   = len(gal_data)
fig = plt.figure(figsize=(24, 5 * N + 0.5), facecolor='k')
gs  = gridspec.GridSpec(
    N, 4, figure=fig,
    width_ratios=[1.45, 1, 1, 1.4],
    hspace=0.05, wspace=0.04,
    left=0.015, right=0.985, top=0.97, bottom=0.06,
)

ax_box = fig.add_subplot(gs[:, 0])   # spans all rows

# ── Full-box panel ────────────────────────────────────────────────────────────
e = ext_box
ax_box.imshow(rgb_box, extent=e, origin='lower', aspect='auto')
ax_box.set_xlim(e[0], e[1]); ax_box.set_ylim(e[2], e[3])
ax_box.set_xticks([]); ax_box.set_yticks([])
for sp in ax_box.spines.values():
    sp.set_edgecolor('w'); sp.set_linewidth(1.2)

ax_box.set_title(
    rf'SIMBA 25 Mpc box  ($z = {z_snap:.2f}$)',
    color='w', fontsize=13, pad=6,
)
add_scalebar(ax_box, e, 5000, fontsize=11)   # 5 Mpc bar

# Mark selected galaxy positions with labeled circles
for gd, col in zip(gal_data, GAL_COLORS):
    dx = gd['center'][0] - box_center_kpc[0]
    dy = gd['center'][1] - box_center_kpc[1]
    ax_box.add_patch(Circle((dx, dy), radius=EXT_BOX * 0.016,
                             lw=1.8, ec=col, fc='none', zorder=5))
    ax_box.text(dx, dy + EXT_BOX * 0.028,
                f'G{gal_data.index(gd)+1}',
                color=col, ha='center', fontsize=10, fontweight='bold', zorder=6)

ax_box.legend(
    handles=[
        Line2D([0], [0], color=cm.magma(0.7), lw=4, label='Gas'),
        Line2D([0], [0], color='0.8',          lw=4, label='Stars'),
    ],
    loc='lower right', fontsize=10,
    framealpha=0.4, facecolor='k', labelcolor='w', edgecolor='0.4',
)

# ── Per-galaxy rows ───────────────────────────────────────────────────────────
for row, (gd, col) in enumerate(zip(gal_data, GAL_COLORS)):
    ax_cgm = fig.add_subplot(gs[row, 1])
    ax_gal = fig.add_subplot(gs[row, 2])
    ax_sed = fig.add_subplot(gs[row, 3])

    gobj  = obj.galaxies[gd['id']]
    mstar = float(gobj.masses['stellar'])
    mdust = float(gobj.masses['dust'])

    # Shared styling for image panels
    for ax in (ax_cgm, ax_gal):
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor(col); sp.set_linewidth(1.5)

    # CGM panel
    e_cgm = gd['ext_cgm']
    ax_cgm.imshow(gd['rgb_cgm'], extent=e_cgm, origin='lower', aspect='auto')
    ax_cgm.set_xlim(e_cgm[0], e_cgm[1]); ax_cgm.set_ylim(e_cgm[2], e_cgm[3])
    ax_cgm.add_patch(Rectangle(
        (-EXT_GALAXY, -EXT_GALAXY), 2*EXT_GALAXY, 2*EXT_GALAXY,
        lw=1.5, ec=col, fc='none', ls='--'))
    add_scalebar(ax_cgm, e_cgm, 50, color=col, fontsize=10)
    ax_cgm.set_title(
        rf'G{row+1}  CGM  ({EXT_CGM} kpc)',
        color=col, fontsize=11, pad=4)

    # Galaxy panel
    e_gal = gd['ext_gal']
    ax_gal.imshow(gd['rgb_gal'], extent=e_gal, origin='lower', aspect='auto')
    ax_gal.set_xlim(e_gal[0], e_gal[1]); ax_gal.set_ylim(e_gal[2], e_gal[3])
    add_scalebar(ax_gal, e_gal, 10, color=col, fontsize=10)
    ax_gal.set_title(
        rf'G{row+1}  ({EXT_GALAXY} kpc)   $M_\star = {mstar:.1e}\,M_\odot$',
        color=col, fontsize=11, pad=4)

    # SED panel
    ax_sed.set_facecolor('#0a0a0a')
    for sp in ax_sed.spines.values(): sp.set_edgecolor('0.45')
    ax_sed.tick_params(colors='w', labelsize=10, which='both', direction='in')
    ax_sed.set_xlabel(r'$\lambda\;[\mu\mathrm{m}]$', color='w', fontsize=12)
    if row == 0:
        ax_sed.set_ylabel(r'$L_\lambda\;[\mathrm{erg\,s^{-1}}]$', color='w', fontsize=12)
    ax_sed.set_title(
        rf'G{row+1}  SED   '
        rf'$M_\mathrm{{dust}}/M_\star = {mdust/mstar:.1e}$'
        rf'   $\mathrm{{SFR}} < 10^{{-10}}\,M_\odot\,\mathrm{{yr}}^{{-1}}$',
        color=col, fontsize=11, pad=4)

    if gd['sed_dust'] is not None:
        wav = gd['sed_wav']
        for inc_row in gd['sed_dust']:
            ax_sed.loglog(wav, inc_row, color=col, lw=0.5, alpha=0.2)
        ax_sed.loglog(wav, gd['sed_dust'].mean(axis=0),
                      color=col, lw=2.5, label='Dust on')
        if gd['sed_nodust'] is not None:
            ax_sed.loglog(wav, gd['sed_nodust'].mean(axis=0),
                          color='w', lw=2.0, ls='--', alpha=0.75, label='Dust off')
        ax_sed.legend(fontsize=10, framealpha=0.3,
                      facecolor='k', labelcolor='w', edgecolor='0.4')
        for bw in [0.15, 0.62, 2.2, 3.6, 24, 100, 250, 850]:
            ax_sed.axvline(bw, color='0.3', lw=0.7, ls=':')
    else:
        ax_sed.text(0.5, 0.5, 'SED not available',
                    transform=ax_sed.transAxes,
                    ha='center', va='center', color='0.5', fontsize=12)
    ax_sed.set_xlim(0.08, 2000)

# ── Save ─────────────────────────────────────────────────────────────────────
out_dir  = os.path.join(os.getcwd(), 'output', 'figures')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f'multiscale_zoom_snap{SNAP}.pdf')
fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='k')
print(f"Saved → {out_path}")
plt.show()

## 8 – Save individual panels

In [ ]:
# ── Save individual figures for each panel ────────────────────────────────────
out_dir = os.path.join(os.getcwd(), 'output', 'figures')
os.makedirs(out_dir, exist_ok=True)

def _fig_image(ax_title, image, ext, bar_kpc, bar_color='w', spine_color='w',
               rect=None, figsize=(6, 6)):
    fig, ax = plt.subplots(figsize=figsize, facecolor='k')
    ax.imshow(image, extent=ext, origin='lower', aspect='equal')
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(spine_color); sp.set_linewidth(1.5)
    if rect is not None:
        ax.add_patch(rect)
    add_scalebar(ax, ext, bar_kpc, color=bar_color, fontsize=12)
    ax.set_title(ax_title, color=spine_color, fontsize=12, pad=5)
    fig.tight_layout(pad=0.3)
    return fig

# ── Full box ──────────────────────────────────────────────────────────────────
fig_box = _fig_image(
    rf'SIMBA 25 Mpc box  ($z = {z_snap:.2f}$)',
    rgb_box, ext_box, bar_kpc=5000, figsize=(7, 7),
)
ax_b = fig_box.axes[0]
for gd, col in zip(gal_data, GAL_COLORS):
    dx = gd['center'][0] - box_center_kpc[0]
    dy = gd['center'][1] - box_center_kpc[1]
    ax_b.add_patch(Circle((dx, dy), radius=EXT_BOX * 0.016,
                           lw=1.8, ec=col, fc='none', zorder=5))
    ax_b.text(dx, dy + EXT_BOX * 0.028, f'G{gal_data.index(gd)+1}',
              color=col, ha='center', fontsize=10, fontweight='bold', zorder=6)
ax_b.legend(
    handles=[
        Line2D([0], [0], color=cm.magma(0.7), lw=4, label='Gas'),
        Line2D([0], [0], color='0.8',          lw=4, label='Stars'),
    ],
    loc='lower right', fontsize=10,
    framealpha=0.4, facecolor='k', labelcolor='w', edgecolor='0.4',
)
p = os.path.join(out_dir, f'box_snap{SNAP}.pdf')
fig_box.savefig(p, dpi=200, bbox_inches='tight', facecolor='k')
print(f"Saved → {p}")
plt.close(fig_box)

# ── Per-galaxy CGM and galaxy panels ─────────────────────────────────────────
for row, (gd, col) in enumerate(zip(gal_data, GAL_COLORS)):
    gobj  = obj.galaxies[gd['id']]
    mstar = float(gobj.masses['stellar'])
    label = f'G{row+1}'

    # CGM
    e_cgm = gd['ext_cgm']
    rect_cgm = Rectangle(
        (-EXT_GALAXY, -EXT_GALAXY), 2*EXT_GALAXY, 2*EXT_GALAXY,
        lw=1.5, ec=col, fc='none', ls='--',
    )
    fig_cgm = _fig_image(
        rf'{label}  CGM  ({EXT_CGM} kpc)  —  face-on',
        gd['rgb_cgm'], e_cgm, bar_kpc=50,
        bar_color=col, spine_color=col, rect=rect_cgm,
    )
    p = os.path.join(out_dir, f'cgm_{label}_snap{SNAP}.pdf')
    fig_cgm.savefig(p, dpi=200, bbox_inches='tight', facecolor='k')
    print(f"Saved → {p}")
    plt.close(fig_cgm)

    # Galaxy
    e_gal = gd['ext_gal']
    fig_gal = _fig_image(
        rf'{label}  ({EXT_GALAXY} kpc)  —  face-on   $M_\star = {mstar:.1e}\,M_\odot$',
        gd['rgb_gal'], e_gal, bar_kpc=10,
        bar_color=col, spine_color=col,
    )
    p = os.path.join(out_dir, f'galaxy_{label}_snap{SNAP}.pdf')
    fig_gal.savefig(p, dpi=200, bbox_inches='tight', facecolor='k')
    print(f"Saved → {p}")
    plt.close(fig_gal)
